In [ ]:
# 1) Install dependencies (run once)
!pip -q install --upgrade sentencepiece

gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-5-codex
gpt-audio-2025-08-28


In [ ]:
# 2) Load a small Qwen model + tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # small instruct variant

# Device/dtype setup
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
model.to(device)
model.eval()

print(f"Loaded {model_name} on {device} with dtype={dtype}")


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded Qwen/Qwen2.5-0.5B-Instruct on cuda with dtype=torch.float16


In [2]:
# 3) Inspect parameters and peek at some weights
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}  (~{total_params/1e6:.1f}M)")
print(f"Trainable parameters: {trainable_params:,}")

# List a few parameter tensors (name, shape)
print("\nSample parameter tensors:")
for i, (name, p) in enumerate(model.named_parameters()):
    if i >= 8:
        break
    print(f" - {name}: {tuple(p.shape)}")

# Inspect the input embedding matrix shape and a small value slice
emb = model.get_input_embeddings().weight.detach().cpu()
print("\nInput embedding weight shape:", tuple(emb.shape))
print("First row, first 10 values:", emb[0, :10].tolist())

# Also peek at another familiar layer (if present)
# This safely grabs the first Linear/LayerNorm weights it finds.
for name, p in model.named_parameters():
    if p.ndim == 2:  # typical Linear weight
        print(f"\nExample linear weight '{name}' shape: {tuple(p.shape)}")
        print("Top-left 3x5 slice:\n", p.detach().cpu()[:3, :5])
        break

Total parameters: 494,032,768  (~494.0M)
Trainable parameters: 494,032,768

Sample parameter tensors:
 - model.embed_tokens.weight: (151936, 896)
 - model.layers.0.self_attn.q_proj.weight: (896, 896)
 - model.layers.0.self_attn.q_proj.bias: (896,)
 - model.layers.0.self_attn.k_proj.weight: (128, 896)
 - model.layers.0.self_attn.k_proj.bias: (128,)
 - model.layers.0.self_attn.v_proj.weight: (128, 896)
 - model.layers.0.self_attn.v_proj.bias: (128,)
 - model.layers.0.self_attn.o_proj.weight: (896, 896)

Input embedding weight shape: (151936, 896)
First row, first 10 values: [-0.0103759765625, 0.040771484375, 0.00970458984375, 6.961822509765625e-05, -0.027099609375, -0.0029754638671875, -0.00115966796875, -0.01953125, 0.0284423828125, -0.006744384765625]

Example linear weight 'model.embed_tokens.weight' shape: (151936, 896)
Top-left 3x5 slice:
 tensor([[-1.0376e-02,  4.0771e-02,  9.7046e-03,  6.9618e-05, -2.7100e-02],
        [-1.4587e-02, -1.3657e-03, -1.7700e-02, -2.6703e-03,  3.7079e-

In [3]:
# 4) Simple inference using the chat template
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short haiku about the moon."},
]

# Build the chat prompt using the model's template
prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer([prompt_text], return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens
gen_ids = outputs[0, inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True))

Glowing in the night sky,
Moonlight bathes the earth in silver glow,
Peace and quiet.


In [4]:
# Quick config summary + shallow module tree
from pprint import pprint

assert "model" in globals(), "Model not found. Run the cell that loads the Qwen model first."

cfg = model.config
interesting = [
    "model_type", "vocab_size", "hidden_size", "intermediate_size",
    "num_hidden_layers", "num_attention_heads", "max_position_embeddings",
    "rope_theta"
]
summary_cfg = {k: getattr(cfg, k) for k in interesting if hasattr(cfg, k)}
print("Model config summary:")
pprint(summary_cfg)

print("\nShallow module tree (first 2 levels):")
def print_children(module, indent=0, max_depth=2):
    if indent > max_depth:
        return
    for name, child in module.named_children():
        print("  " * indent + f"- {name}: {child.__class__.__name__}")
        print_children(child, indent + 1, max_depth)

print_children(model, max_depth=2)

# ... rest of notebook ...

Model config summary:
{'hidden_size': 896,
 'intermediate_size': 4864,
 'max_position_embeddings': 32768,
 'model_type': 'qwen2',
 'num_attention_heads': 14,
 'num_hidden_layers': 24,
 'rope_theta': 1000000.0,
 'vocab_size': 151936}

Shallow module tree (first 2 levels):
- model: Qwen2Model
  - embed_tokens: Embedding
  - layers: ModuleList
    - 0: Qwen2DecoderLayer
    - 1: Qwen2DecoderLayer
    - 2: Qwen2DecoderLayer
    - 3: Qwen2DecoderLayer
    - 4: Qwen2DecoderLayer
    - 5: Qwen2DecoderLayer
    - 6: Qwen2DecoderLayer
    - 7: Qwen2DecoderLayer
    - 8: Qwen2DecoderLayer
    - 9: Qwen2DecoderLayer
    - 10: Qwen2DecoderLayer
    - 11: Qwen2DecoderLayer
    - 12: Qwen2DecoderLayer
    - 13: Qwen2DecoderLayer
    - 14: Qwen2DecoderLayer
    - 15: Qwen2DecoderLayer
    - 16: Qwen2DecoderLayer
    - 17: Qwen2DecoderLayer
    - 18: Qwen2DecoderLayer
    - 19: Qwen2DecoderLayer
    - 20: Qwen2DecoderLayer
    - 21: Qwen2DecoderLayer
    - 22: Qwen2DecoderLayer
    - 23: Qwen2DecoderL

In [7]:
!pip install torchinfo
# Tabular architecture summary (layers, shapes, params) via torchinfo
import torch
from torchinfo import summary

# Prepare a small dummy batch to let torchinfo trace shapes
seq_len = 32
dummy = tokenizer(
    "Hello world",
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=seq_len,
)
dummy = {k: v.to(model.device) for k, v in dummy.items()}

summ = summary(
    model,
    input_data=dummy,                 # pass dict(inputs) for HF models
    depth=3,                          # increase for more nesting
    col_names=("input_size", "output_size", "num_params", "params_percent"),
    row_settings=("var_names",),
    verbose=0,
)
print(summ)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Layer (type (var_name))                                      Input Shape               Output Shape              Param #                   Param %
Qwen2ForCausalLM (Qwen2ForCausalLM)                          --                        --                        --                        -21.60%
├─Qwen2Model (model)                                         --                        --                        --                             --
│    └─Embedding (embed_tokens)                              [1, 32]                   [1, 32, 896]              136,134,656                21.60%
│    └─Qwen2RotaryEmbedding (rotary_emb)                     [1, 32, 896]              [1, 32, 64]               --                             --
│    └─ModuleList (layers)                                   --                        --                        --                             --
│    │    └─Qwen2DecoderLayer (0)                            [1, 32, 896]              [1, 32, 896]              14,91

In [9]:
!pip install torchview graphviz
# Visual DAG of the model using torchview (renders via graphviz)
# Note: This produces a large graph for LLMs; keep depth small (e.g., 1–2).
from torchview import draw_graph

graph = draw_graph(
    model,
    input_data=dummy,     # same dict input
    expand_nested=True,   # show nested submodules
    depth=2,              # increase to see deeper layers, but it gets big fast
    roll=True,            # compact repeated modules
)

# In most notebooks this will display inline (SVG)
graph.visual_graph

# To save as PNG instead, uncomment:
# graph.visual_graph.render("qwen_architecture", format="png", cleanup=True)
# from IPython.display import Image
# Image("qwen_architecture.png")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchview]


KeyError: '135469714605104'

In [ ]:
print(model)
'''
'hidden_size': 896,
'intermediate_size': 4864,
'max_position_embeddings': 32768, (=sequence length)
'model_type': 'qwen2',
'num_attention_heads': 14,
'num_hidden_layers': 24,
'rope_theta': 1000000.0,
'vocab_size': 151936}
'''

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [11]:
import inspect
from transformers.models.qwen2 import modeling_qwen2 as qwen2_impl

print("Source file:", inspect.getfile(qwen2_impl))
print("\nHas classes:",
      hasattr(qwen2_impl, "Qwen2RotaryEmbedding"),
      hasattr(qwen2_impl, "Qwen2Attention"))

# Peek at the RoPE module
print("\nQwen2RotaryEmbedding (truncated):\n")
print(inspect.getsource(qwen2_impl.Qwen2RotaryEmbedding)[:1200])

# Peek at where it’s applied in attention (truncated)
print("\nQwen2Attention.forward (truncated):\n")
print(inspect.getsource(qwen2_impl.Qwen2Attention.forward)[:1200])

Source file: /opt/conda/lib/python3.10/site-packages/transformers/models/qwen2/modeling_qwen2.py

Has classes: True True

Qwen2RotaryEmbedding (truncated):

class Qwen2RotaryEmbedding(nn.Module):
    def __init__(self, config: Qwen2Config, device=None):
        super().__init__()
        # BC: "rope_type" was originally "type"
        if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
            self.rope_type = config.rope_scaling.get("rope_type", config.rope_scaling.get("type"))
        else:
            self.rope_type = "default"
        self.max_seq_len_cached = config.max_position_embeddings
        self.original_max_seq_len = config.max_position_embeddings

        self.config = config
        self.rope_init_fn = ROPE_INIT_FUNCTIONS[self.rope_type]

        inv_freq, self.attention_scaling = self.rope_init_fn(self.config, device)
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.original_inv_freq = self.inv_freq

    def _dyna

In [12]:
print("rope_theta:", model.config.rope_theta)  # e.g., 1_000_000.0
print("max_position_embeddings:", model.config.max_position_embeddings)
print("rope_scaling:", getattr(model.config, "rope_scaling", None))

# The RoPE module on the model
print(model.model.rotary_emb)

rope_theta: 1000000.0
max_position_embeddings: 32768
rope_scaling: None
Qwen2RotaryEmbedding()


In [14]:
# let's drop one decoder layer and check the model still works
import torch.nn as nn

def drop_decoder_layer_inplace(model, drop_idx: int):
    """
    Remove one Qwen2DecoderLayer at index `drop_idx` from model.model.layers
    and update model.config.num_hidden_layers accordingly.
    """
    layers = model.model.layers
    n = len(layers)
    assert 0 <= drop_idx < n, f"drop_idx must be in [0, {n-1}]"
    # Build a new ModuleList without the dropped layer
    new_layers = nn.ModuleList([layer for i, layer in enumerate(layers) if i != drop_idx])
    model.model.layers = new_layers
    model.config.num_hidden_layers = len(new_layers)
    return model

# Example: drop the 13th layer (index 12). You can choose any index [0..23].
drop_idx = 12
model = drop_decoder_layer_inplace(model, drop_idx)

# Quick sanity check: hidden states count = num_hidden_layers + 1 (includes embeddings)
inputs = tokenizer("Hello world", return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)
print("Layers after drop:", len(model.model.layers))
print("Config.num_hidden_layers:", model.config.num_hidden_layers)
print("Hidden states returned:", len(out.hidden_states))


Layers after drop: 22
Config.num_hidden_layers: 22
Hidden states returned: 23


In [16]:
# 4) Simple inference using the chat template
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short haiku about the moon."},
]

# Build the chat prompt using the model's template
prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer([prompt_text], return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens
gen_ids = outputs[0, inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True))

Moonlight falls,
Moonlight shines bright,  
Silent whispers.

The night air is soft and peaceful,
Moonlight reflects off a lake,
Silent whispers.


In [17]:
import inspect
from transformers.models.qwen2 import modeling_qwen2 as qwen2_impl

print("Source:", inspect.getfile(qwen2_impl))

src = inspect.getsource(qwen2_impl.Qwen2Model.forward)
snippet_start = src.find("if position_ids is None")
print(src[snippet_start: snippet_start + 1200])  # shows how position_ids is built

attn_src = inspect.getsource(qwen2_impl.Qwen2Attention.forward)
print("\nWhere it's used (truncated):\n")
print(attn_src[:1200])  # shows call to apply_rotary_pos_emb(...)

Source: /opt/conda/lib/python3.10/site-packages/transformers/models/qwen2/modeling_qwen2.py
if position_ids is None:
            position_ids = cache_position.unsqueeze(0)

        causal_mask = self._update_causal_mask(
            attention_mask, inputs_embeds, cache_position, past_key_values, output_attentions
        )

        hidden_states = inputs_embeds

        # create position embeddings to be shared across the decoder layers
        position_embeddings = self.rotary_emb(hidden_states, position_ids)

        # decoder layers
        all_hidden_states = () if output_hidden_states else None
        all_self_attns = () if output_attentions else None

        for decoder_layer in self.layers[: self.config.num_hidden_layers]:
            if output_hidden_states:
                all_hidden_states += (hidden_states,)

            if self.gradient_checkpointing and self.training:
                layer_outputs = self._gradient_checkpointing_func(
                    decoder_layer.__c

In [18]:
import torch
from transformers.models.qwen2 import modeling_qwen2 as qwen2_impl

# Monkey-patch to capture position_ids passed into RoPE
captured = {}
_orig = qwen2_impl.apply_rotary_pos_emb

def _spy_apply_rotary_pos_emb(*args, **kwargs):
    position_ids = kwargs.get("position_ids", None)
    if position_ids is None and len(args) >= 5:
        position_ids = args[4]
    if position_ids is not None:
        captured['position_ids'] = position_ids.detach().cpu()
    return _orig(*args, **kwargs)

qwen2_impl.apply_rotary_pos_emb = _spy_apply_rotary_pos_emb

# Run a forward pass
inputs = tokenizer("Hello world", return_tensors="pt").to(model.device)
with torch.no_grad():
    _ = model(**inputs)

# See the indices (shape: [batch, seq_len])
print("position_ids used:\n", captured.get('position_ids'))

# Restore original to avoid side effects
qwen2_impl.apply_rotary_pos_emb = _orig

position_ids used:
 None


In [19]:
# Build custom position_ids (e.g., start at 1000, or reset after padding)
inputs = tokenizer("Hello world", return_tensors="pt").to(model.device)
b, s = inputs.input_ids.shape
custom_pos = torch.arange(1000, 1000 + s, device=model.device).unsqueeze(0).expand(b, -1)

with torch.no_grad():
    out = model(**inputs, position_ids=custom_pos)

In [20]:
# Let's do an exercise and reverse the positional encodings so instead of [0...n], we use [n..0]
# Great exercise. You don’t need to touch RoPE itself—just feed reversed position_ids to the model so the attention layers index cos/sin in reverse.

import torch

def make_reversed_position_ids(input_ids, attention_mask=None, past_key_values_length=0):
    """
    Build descending position ids per example, turning [0..n] into [n..0].
    Handles padding (via attention_mask) and past_key_values_length.
    """
    device = input_ids.device
    b, s = input_ids.shape

    if attention_mask is None:
        # Standard ascending: [p, p+1, ..., p+s-1]
        std = torch.arange(past_key_values_length, past_key_values_length + s, device=device)
        std = std.unsqueeze(0).expand(b, -1)
    else:
        # Standard ascending with padding-aware cumsum
        std = (attention_mask.cumsum(-1) - 1).clamp(min=0)
        std = std[:, -s:]  # keep only the current segment
        if past_key_values_length:
            std = std + past_key_values_length

    # Reverse per example: max_pos - pos
    max_pos = std.amax(dim=-1, keepdim=True)
    rev = (max_pos - std).to(dtype=torch.long)
    return rev

# Example: single forward pass with reversed positions (disable cache to avoid mismatch)
inputs = tokenizer("Hello world", return_tensors="pt").to(model.device)
rev_pos = make_reversed_position_ids(inputs.input_ids, inputs.get("attention_mask"), past_key_values_length=0)

with torch.no_grad():
    out = model(**inputs, position_ids=rev_pos, use_cache=False)

print("Reversed position_ids:\n", rev_pos)

Reversed position_ids:
 tensor([[1, 0]], device='cuda:0')


In [21]:
# 4) Simple inference using the chat template
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short haiku about the moon."},
]

# Build the chat prompt using the model's template
prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer([prompt_text], return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens
gen_ids = outputs[0, inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True))

The moon whispers:
In the night sky,
Softly sings the stars.
Moonlight's soft glow,
Whispers peace.

Does that do it for you?
